# Uplift modeling on the Criteo Uplift dataset — reproduce all results

**How to use:** run the cells **one at a time, top to bottom** (click ▶ on each cell and wait until it finishes).
Everything is saved to your Google Drive in `MyDrive/uplift-criteo-run/`, so if Colab disconnects
you can simply run the cells again from the top: finished work is detected and skipped.

Expected time on a free Colab CPU runtime: about 45–75 minutes in total (download 1–3 min, prepare ~5 min,
training ~30–50 min of which the causal forest is ~15–20 min, evaluation ~5 min).

## Step 1 — Connect Google Drive
A pop-up asks for permission to use your Drive: allow it.

In [ ]:
import os
SYNTHETIC_TEST = int(os.environ.get("SYNTHETIC_TEST", "0"))   # 0 = real Criteo data (keep 0)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/uplift-criteo-run'
    CODE_DIR = '/content/uplift-criteo'
except ImportError:                      # running outside Colab (used only for testing)
    RUN_DIR = os.path.abspath(os.environ.get("RUN_DIR", "uplift-criteo-run"))
    CODE_DIR = os.path.abspath(os.environ.get("CODE_DIR", "uplift-criteo"))
os.makedirs(RUN_DIR, exist_ok=True)
print('Results will be saved in:', RUN_DIR)

## Step 2 — Download the code and install the exact package versions
Takes 1–3 minutes. If Colab asks you to *restart the session*, click **Restart**, then run Step 1 again and continue with Step 3.

In [ ]:
REPO = 'https://github.com/Akash-420bit/uplift-criteo'
if not os.path.exists(CODE_DIR):
    !git clone -q {REPO} {CODE_DIR}
%cd {CODE_DIR}
!pip install -q -r requirements.txt
print('Code ready in', CODE_DIR)

## Step 3 — Download the data, draw the 2M-row sample, split it, run the randomization check
The 311 MB file is downloaded once into Drive and its checksum is verified. About 5 minutes.

In [ ]:
SYN = f'--synthetic {SYNTHETIC_TEST}' if SYNTHETIC_TEST else ''
COMMON = f'--root "{RUN_DIR}" --results "{RUN_DIR}/results" {SYN}'
if not os.path.exists(f'{RUN_DIR}/data/sample_split.npz'):
    !python -m src.pipeline --step prepare {COMMON}
else:
    print('Sample already prepared - skipping.')

## Step 4 — Train the four models for both outcomes
Each finished model is saved immediately. If Colab disconnects, re-run Steps 1–2 and then this cell: it continues where it stopped.

In [ ]:
!python -m src.pipeline --step train {COMMON}

## Step 5 — Evaluate on the test split (Qini, top-k uplift, 200 bootstrap resamples)

In [ ]:
!python -m src.pipeline --step evaluate {COMMON}

## Step 6 — Budget simulation (targeting fraction chosen on validation, profit on test)

In [ ]:
!python -m src.pipeline --step budget {COMMON}

## Step 7 — Figures and LaTeX tables

In [ ]:
!python -m src.pipeline --step report {COMMON}
import pandas as pd
print(pd.read_csv(f'{RUN_DIR}/results/main_results.csv').query("metric=='auqc'").to_string(index=False))

## Step 8 — Zip the results folder
A file `results.zip` appears in `MyDrive/uplift-criteo-run/` (and downloads automatically in Colab).

In [ ]:
import shutil
zip_path = shutil.make_archive(f'{RUN_DIR}/results', 'zip', RUN_DIR, 'results')
print('Created', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass